# Harris County, TX — building depths from NFHL + DEM (Phase 1, step 4)

Fourth link. Join the three local layers built so far:

- buildings + per-building ground elevation (notebooks 01, 03)
- NFHL flood hazard zones with `STATIC_BFE` (notebook 02)
- NFHL `S_BFE` lines for zones where `STATIC_BFE` is missing (notebook 02)

and produce a per-building `depth_100_m` and `depth_500_m`.

**Why two BFE sources.** Coastal/Zone-AE polygons in `S_FLD_HAZ_AR` carry a
`STATIC_BFE` attribute set; riverine A and AE-without-static zones leave
it as 0/NaN and instead reference a separate `S_BFE` line layer. We use
`STATIC_BFE` where present, fall back to the nearest `S_BFE` line
elevation otherwise, and only mark a building unscored if neither exists
(zones X, D, ANI).

**500-yr depth.** NFHL is dual-frequency (1% and 0.2%) at the polygon
level but ships a BFE only for the 1% event. For the 500-yr depth we use
the HAZUS shadow assumption: SFHA buildings get `depth_100 + 1 ft`, and
shaded-X buildings (in the 0.2% footprint but outside SFHA) get a flat
1 ft above ground. Both are documented approximations and replaced in
Phase 2 by an actual 500-yr WSE grid.

**Units.** All depths and elevations stored in meters. NFHL ELEV/STATIC_BFE
are NAVD88 feet — converted on read with `× 0.3048`.

**Inputs:** parquets from notebooks 01–03.

**Output:** `data/raw/harris_building_depths.parquet`


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd


In [ ]:
FT_TO_M = 0.3048
SFHA_ZONES = {"A", "AE", "AH", "AO", "AR", "A99", "V", "VE"}
DEPTH_500_OVER_100_M = 1.0 * FT_TO_M  # HAZUS shadow assumption: 500-yr WSE ≈ 100-yr + 1 ft.
DEPTH_500_X_SHADED_M = 1.0 * FT_TO_M  # nominal depth assigned to 0.2%-shaded X polygons.

REPO_ROOT = Path.cwd().resolve().parents[1]
RAW_DIR = REPO_ROOT / "data" / "raw"
BUILDINGS_PATH = RAW_DIR / "harris_buildings.parquet"
ELEV_PATH = RAW_DIR / "harris_building_elev.parquet"
ZONES_PATH = RAW_DIR / "harris_nfhl_zones.parquet"
BFE_PATH = RAW_DIR / "harris_nfhl_bfe.parquet"
OUT_PATH = RAW_DIR / "harris_building_depths.parquet"
OUT_PATH


## Load the four inputs and align CRS

All four are already EPSG:4326 from earlier notebooks; the explicit
`set_crs` is just defensive.


In [ ]:
buildings = gpd.read_parquet(BUILDINGS_PATH).set_crs("EPSG:4326", allow_override=True)
buildings = buildings[["id", "class", "subtype", "num_floors", "height", "geometry"]]
buildings["centroid"] = buildings.geometry.centroid
buildings = buildings.set_geometry("centroid")

elev = pd.read_parquet(ELEV_PATH)
buildings = buildings.merge(elev, on="id", how="left")

zones = gpd.read_parquet(ZONES_PATH).set_crs("EPSG:4326", allow_override=True)
bfe = gpd.read_parquet(BFE_PATH).set_crs("EPSG:4326", allow_override=True)
len(buildings), len(zones), len(bfe)


## Spatial-join centroids to zones

We use the **centroid** for the join, not the footprint, to avoid
duplicating buildings that happen to straddle a zone boundary. A small
fraction of footprints actually cross zones; for those the centroid rule
is the cleanest tie-break and matches what HAZUS does at the parcel
level. Zones are not mutually exclusive in raw NFHL (overlapping AE +
shaded-X polygons) — keep the most severe by ranking SFHA > X-shaded > X.


In [ ]:
ZONE_RANK = {z: 3 for z in SFHA_ZONES}
ZONE_RANK["X"] = 1  # base; we'll bump shaded-X to 2 below.

zones_in = zones[["FLD_ZONE", "ZONE_SUBTY", "STATIC_BFE", "SFHA_TF", "geometry"]].copy()
zones_in["_rank"] = zones_in["FLD_ZONE"].map(ZONE_RANK).fillna(0).astype(int)
is_shaded = zones_in["ZONE_SUBTY"].fillna("").str.contains("0.2 PCT", case=False, regex=False)
zones_in.loc[is_shaded & (zones_in["_rank"] < 2), "_rank"] = 2

joined = gpd.sjoin(buildings, zones_in, how="left", predicate="within")
# A centroid that lands in two overlapping zones gets two rows; keep the most severe.
joined = joined.sort_values("_rank", ascending=False).drop_duplicates("id", keep="first")
joined["FLD_ZONE"] = joined["FLD_ZONE"].fillna("OUT")
joined.groupby("FLD_ZONE").size().sort_values(ascending=False).head(10)


## Resolve BFE per building

1. Use `STATIC_BFE` (feet → meters) where it's > 0.
2. Otherwise, for SFHA buildings, snap to the nearest `S_BFE` line and
   take its `ELEV` value.
3. Anything still unresolved leaves `bfe_m = NaN` and depths stay NaN —
   the EAD notebook treats those as zero damage and flags them.


In [ ]:
bfe_m = pd.Series(np.nan, index=joined.index, dtype="float64")
static = pd.to_numeric(joined["STATIC_BFE"], errors="coerce")
has_static = static > 0
bfe_m.loc[has_static] = static.loc[has_static].astype("float64") * FT_TO_M
print(f"static BFE: {int(has_static.sum()):,}")

needs_lookup = joined["FLD_ZONE"].isin(SFHA_ZONES) & ~has_static
if needs_lookup.any():
    targets = joined.loc[needs_lookup, ["id", "centroid"]].set_geometry("centroid").to_crs("EPSG:3857")
    bfe_proj = bfe[["ELEV", "geometry"]].to_crs("EPSG:3857")
    nearest = gpd.sjoin_nearest(targets, bfe_proj, how="left", distance_col="_d")
    nearest = nearest.drop_duplicates("id", keep="first")
    elev_ft = pd.to_numeric(nearest["ELEV"], errors="coerce")
    fill = (elev_ft * FT_TO_M).to_numpy()
    bfe_m.loc[needs_lookup] = fill
    print(f"line-interpolated BFE: {int(needs_lookup.sum()):,} (max snap distance {nearest['_d'].max():.0f} m)")

joined["bfe_m"] = bfe_m.values
int(joined["bfe_m"].notna().sum()), int(joined["bfe_m"].isna().sum())


## Compute 100-yr and 500-yr depths

`depth_100_m = max(0, BFE_m − elev_m)` for any building with a BFE; zero
elsewhere. The 500-yr value follows the rules described in the intro: a
1-ft adder over the 100-yr depth inside SFHA, a flat 1-ft above ground in
shaded-X, otherwise zero.


In [ ]:
elev_m = pd.to_numeric(joined["elev_m"], errors="coerce")
in_sfha = joined["FLD_ZONE"].isin(SFHA_ZONES)
is_shaded_x = (joined["FLD_ZONE"] == "X") & joined["ZONE_SUBTY"].fillna("").str.contains(
    "0.2 PCT", case=False, regex=False
)

depth_100 = (joined["bfe_m"] - elev_m).clip(lower=0).fillna(0.0)
depth_500 = pd.Series(0.0, index=joined.index)
depth_500.loc[in_sfha] = depth_100.loc[in_sfha] + DEPTH_500_OVER_100_M
depth_500.loc[is_shaded_x] = DEPTH_500_X_SHADED_M

joined["depth_100_m"] = depth_100.astype("float32")
joined["depth_500_m"] = depth_500.astype("float32")
joined["in_sfha"] = in_sfha
joined["is_shaded_x"] = is_shaded_x
joined[["depth_100_m", "depth_500_m"]].describe()


In [ ]:
out_cols = [
    "id", "class", "subtype", "num_floors", "height",
    "FLD_ZONE", "ZONE_SUBTY", "in_sfha", "is_shaded_x",
    "elev_m", "bfe_m", "depth_100_m", "depth_500_m",
]
out = pd.DataFrame(joined[out_cols]).reset_index(drop=True)
out.to_parquet(OUT_PATH, compression="zstd")
print(f"wrote {OUT_PATH} — {len(out):,} rows")
print(
    f"  in SFHA:      {int(out['in_sfha'].sum()):,}\n"
    f"  shaded-X:     {int(out['is_shaded_x'].sum()):,}\n"
    f"  depth_100>0:  {int((out['depth_100_m'] > 0).sum()):,}"
)
out.head(5)


## Next

Move to `05_harris_county_ead.ipynb`: assign HAZUS archetypes from
Overture `class`/`subtype`, look up depth-damage curves, compute dollar
damage at each return period, and integrate to Expected Annual Damage.
